# Results viewer

Run by hand, cell by cell, after an experiment has finished. It only **reads** a run
folder (`<runs_dir>/<RUN>/`, from `configs/paths.yaml`) and writes nothing.

1. Did the run finish, and how long did each stage take?
2. Did the language model learn? (notebook 03: loss curves, loss per field)
3. Fraud detection: every model on val and test, with the noise band
4. Results per month of the test period
5. All finished runs side by side, compared to `base`

> **Results are private.** Before committing this notebook, clear its outputs
> (Jupyter: *Edit → Clear All Outputs*, or
> `jupyter nbconvert --clear-output --inplace 06_results.ipynb`).

In [ ]:
# ======================================================================
# EDIT HERE: which run to look at, and the reference to compare with.
# ======================================================================
RUN = "base"
REFERENCE = "base"
# ======================================================================

import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from IPython.display import display

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.experiment import runs_root

RUNS = runs_root()
R = RUNS / RUN
assert R.exists(), f"{R} not found. Runs available: {sorted(p.name for p in RUNS.iterdir() if p.is_dir())}"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def fmt(df, formats):
    # Formatted copy for display: {column: "{:.3f}"}; missing values shown as "-".
    out = df.copy()
    for col, f in formats.items():
        if col in out.columns:
            out[col] = out[col].map(lambda v, f=f: "-" if pd.isna(v) else f.format(v))
    return out



def metrics(run):
    return json.loads((RUNS / run / "outputs" / "metrics.json").read_text())


# Runs shown in the comparison plots: the reference first, then the run.
SHOWN = list(dict.fromkeys([REFERENCE, RUN]))
ARMS = ["raw", "embeddings", "combined"]

print(f"Run folder: {R}")

## 1. Run status

Every stage should say `done`.

In [ ]:
info = json.loads((R / "run_info.json").read_text())
print(f"{info.get('name')}: {info.get('status')}")

stages = pd.DataFrame(info.get("stages", {})).T
if "seconds" in stages:
    stages["duration"] = pd.to_timedelta(pd.to_numeric(stages["seconds"], errors="coerce").fillna(0), unit="s").astype(str)
stages[[c for c in ("status", "duration", "error") if c in stages]]

In [ ]:
# What this experiment changes (its frozen experiment.yaml).
print((R / "experiment.yaml").read_text())

## 2. Did the language model learn? (notebook 03)

Read from the executed notebook 03 of the run. What to look for:

- **Loss well below the dashed line** `ln(V)` (the loss of a model that guesses at
  random). Barely below it = the model learned little.
- **Validation loss going down, then flattening.** Going back up = overfitting; the best
  checkpoint (lowest val loss) is the one used for the embeddings.
- **Per field:** a field far below its own `ln(vocab)` is one the model predicts well.

In [ ]:
# Preferred: the history notebook 03 saves as data (runs made after 2026-10-30).
# Older runs (e.g. the first base): read back from notebook 03's printed output.
hist_path = R / "model" / "training_history.json"
field_rows = None
if hist_path.exists():
    H = json.loads(hist_path.read_text())
    source = hist_path.name
    train = list(zip(H["steps"], H["train_loss"]))
    val = list(zip(H["val_steps"], H["val_loss"]))
    baseline = H.get("baseline_loss")
    if H.get("field_loss"):
        names = list(H.get("field_names") or [])
        vocab = H.get("field_vocab") or {}
        field_rows = []
        for j, (loss, n) in enumerate(zip(H["field_loss"], H["field_loss_counts"])):
            name = names[j] if j < len(names) else "<sep>/<eos>"
            rand = float(np.log(vocab[name])) if vocab.get(name, 0) > 1 else np.nan
            field_rows.append({"field": name, "val loss": loss if n else np.nan,
                               "random guess ln(vocab)": rand, "targets": int(n)})
else:
    source = "notebook 03 printed output (no training_history.json in this run)"
    nb03 = R / "notebooks" / "03_foundation_model_training.ipynb"
    text = ""
    if nb03.exists():
        for cell in json.loads(nb03.read_text(encoding="utf-8"))["cells"]:
            for out in cell.get("outputs", []):
                if out.get("output_type") == "stream":
                    t = out.get("text", "")
                    text += "".join(t) if isinstance(t, list) else t
    train = [(int(s), float(l)) for s, l in re.findall(r"step\s+(\d+)/\d+\s+loss ([\d.]+)", text)]
    val = [(int(s), float(l)) for s, l in re.findall(r"step\s+(\d+)/\d+\s+VAL loss ([\d.]+)", text)]
    m = re.search(r"ln\s*\(?V\)?\s*=\s*([\d.]+)", text)
    baseline = float(m.group(1)) if m else None
    block = re.search(r"Validation loss by field position.*?(?=\n\s*\(a field|\Z)", text, re.S)

print(f"Read from: {source}")
if train or val:
    fig, ax = plt.subplots(figsize=(9, 3.6))
    if train:
        steps, losses = zip(*train)
        ax.plot(steps, losses, lw=0.6, alpha=0.35, c="tab:blue")
        ax.plot(steps, pd.Series(losses).rolling(20, min_periods=1).mean(), lw=1.6,
                c="tab:blue", label="train (smoothed)")
    if val:
        ax.plot(*zip(*val), "o-", lw=1.6, c="tab:orange", label="validation")
    if baseline:
        ax.axhline(baseline, ls="--", c="gray", label=f"random guess ln(V) = {baseline:.2f}")
    ax.set_xlabel("training step")
    ax.set_ylabel("loss")
    ax.set_title(f"{RUN}: next-token loss")
    ax.legend()
    plt.show()
    if val:
        best = min(val, key=lambda p: p[1])
        print(f"best validation loss {best[1]:.3f} at step {best[0]}  "
              f"(last: {val[-1][1]:.3f} at step {val[-1][0]})")
else:
    print("No training history found.")

In [ ]:
# RUN vs REFERENCE: validation loss. Comparable only when both use the same tokenizer
# (same fields and vocabulary). from_run runs reuse their source's model: no curve.
if len(SHOWN) > 1:
    fig, ax = plt.subplots(figsize=(9, 3.6))
    for run in SHOWN:
        h_path = RUNS / run / "model" / "training_history.json"
        if h_path.exists():
            h = json.loads(h_path.read_text())
            ax.plot(h["val_steps"], h["val_loss"], "o-", label=run)
        else:
            print(f"{run}: no training_history.json")
    ax.set_xlabel("training step")
    ax.set_ylabel("validation loss")
    ax.legend()
    plt.show()

In [ ]:
# Loss per field (validation, end of training): how well the model predicts each field.
# "learned" = how far below a random guess, as a share of the random guess.
if field_rows:
    fields = pd.DataFrame(field_rows).set_index("field").drop(columns="targets")
    fields["learned"] = 1 - fields["val loss"] / fields["random guess ln(vocab)"]
    fields = fields.sort_values("learned", ascending=False)
    display(fmt(fields, {"val loss": "{:.3f}", "random guess ln(vocab)": "{:.2f}", "learned": "{:.0%}"}))

    shown = fields.dropna(subset=["val loss"]).sort_values("val loss")
    fig, ax = plt.subplots(figsize=(8, 0.35 * len(shown) + 1.2))
    y = np.arange(len(shown))
    ax.barh(y, shown["val loss"], color="tab:blue", label="validation loss")
    ax.plot(shown["random guess ln(vocab)"], y, "|", ms=14, mew=2, c="crimson", label="random guess")
    ax.set_yticks(y, shown.index)
    ax.invert_yaxis()
    ax.set_xlabel("loss (lower = better predicted)")
    ax.legend(loc="lower right")
    plt.show()
elif hist_path.exists():
    print("No per-field loss recorded in this run.")
elif block:
    print(block.group(0))

## 3. Fraud detection

One row per model ("arm"), all trained and scored on the same transfers:

| Arm | Input |
|---|---|
| `raw` | the original columns (the baseline) |
| `embeddings` | the language model's embedding of the last transfers only |
| `combined` | both |
| `logreg_*` | the same three inputs with logistic regression instead of XGBoost |

**How to read it**

- **AUC**: 0.5 = chance, 1.0 = perfect. `± se` is the noise; two numbers closer than
  about 2 × se of each other are *the same*.
- **train AUC far above test AUC** = the model memorised the training data.
- **AP** (average precision): chance = the fraud rate. Read it next to AUC: with few frauds
  AUC looks good easily, AP shows how clean the top of the ranking really is.
- **lift** = AP / fraud rate: 1 = chance, 3 = three times better than flagging at random.
- **recall @1% FPR**: share of frauds caught when 1% of legitimate transfers are flagged.
- **frauds / fraud value caught top 1%**: flag the 1% highest scores; share of the frauds
  (by count / by amount) among them.

**Plots below** (light = `REFERENCE`, dark = `RUN`):

- **Bars:** each arm's test AUC (above chance = above 0.5), test AP (above chance = above the
  fraud rate) and share of frauds caught in
  the top 1%. Bars whose error bars overlap are not different.
- **Curve:** share of frauds caught (y) when flagging the highest-scored X% of transfers
  (x). Higher = more frauds per alert; the dotted line is flagging at random.

In [ ]:
M = json.loads((R / "outputs" / "metrics.json").read_text())
print(f"test frauds: {M.get('n_test_positives')}   fraud rate (test): {M.get('prevalence_test', float('nan')):.4%}\n")

COLS = {
    "test_auc": "test AUC", "test_auc_se": "± se", "test_ap": "test AP", "train_auc": "train AUC",
    "test_lift": "test lift", "test_recall_at_1pct_fpr": "recall @1% FPR",
    "test_recall_at_top_1pct": "frauds caught top 1%",
    "test_value_recall_at_top_1pct": "fraud value caught top 1%",
}
arms = pd.DataFrame(M["arms"]).T
table = arms[[c for c in COLS if c in arms]].astype(float).rename(columns=COLS)

def verdict(row):
    gap = row["test AUC"] - 0.5
    return "above chance" if gap > 2 * row["± se"] else "at chance (within noise)"

table["vs chance"] = table.apply(verdict, axis=1)
fmt(table, {c: ("{:.1%}" if any(w in c for w in ("caught", "recall")) else "{:.3f}")
            for c in table.columns if c != "vs chance"})

In [ ]:
# Test set: share of fraud per transfer, per customer and per account.
# A customer (account) counts as fraud if at least one of its test transfers is fraud.
from src.experiment import Experiment

SCH = json.loads(Experiment.load(R / "experiment.yaml", validate=False).paths()["SCHEMA_PATH"].read_text())
CUST, ACC = SCH["customer_id"], SCH["account_number"]
test = pd.read_parquet(R / "embeddings" / "test_features.parquet",
                       columns=["__row_id__", CUST, ACC, "_target"])

rows = {}
for level, key in (("transfer", None), ("customer", CUST), ("account", ACC)):
    y = test["_target"] if key is None else test.groupby(key)["_target"].max()
    rows[level] = {"count": len(y), "fraud": int(y.sum()), "fraud rate": y.mean()}
fmt(pd.DataFrame(rows).T, {"count": "{:,.0f}", "fraud": "{:,.0f}", "fraud rate": "{:.4%}"})

In [ ]:
# Is a model better than raw features beyond the noise? The gap must exceed ~2 x the
# combined standard error (conservative: both models score the same transfers).
rows = []
for base_arm, others in (("raw", ("embeddings", "combined")),
                         ("logreg_raw", ("logreg_embeddings", "logreg_combined"))):
    if base_arm not in M["arms"]:
        continue
    b = M["arms"][base_arm]
    for arm in others:
        if arm not in M["arms"]:
            continue
        a = M["arms"][arm]
        gap = a["test_auc"] - b["test_auc"]
        noise = 2 * np.hypot(a["test_auc_se"], b["test_auc_se"])
        rows.append({"model": arm, "vs": base_arm, "test AUC gap": gap, "noise band (±)": noise,
                     "reading": ("better" if gap > noise else "worse" if gap < -noise
                                 else "no difference")})
fmt(pd.DataFrame(rows), {"test AUC gap": "{:+.3f}", "noise band (±)": "{:.3f}"})

In [ ]:
# The XGBoost arms side by side; REFERENCE (light) next to RUN (dark).
x = np.arange(len(ARMS))
w = 0.8 / len(SHOWN)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, run in enumerate(SHOWN):
    t = pd.DataFrame(metrics(run)["arms"]).T.reindex(ARMS).astype(float)
    style = dict(width=w, label=run, color="tab:blue" if run == RUN else "lightsteelblue")
    axes[0].bar(x + i * w, t["test_auc"] - 0.5, bottom=0.5, yerr=2 * t["test_auc_se"],
                capsize=4, **style)
    axes[1].bar(x + i * w, t["test_ap"], **style)
    axes[2].bar(x + i * w, t["test_recall_at_top_1pct"], **style)
axes[0].set_title("test AUC (error bar = 2 x se; 0.5 = chance)")
axes[1].axhline(metrics(RUN)["prevalence_test"], ls="--", c="gray", label="chance (fraud rate)")
axes[1].set_title("test AP")
axes[2].axhline(0.01, ls="--", c="gray", label="random (1%)")
axes[2].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[2].set_title("frauds caught in the top 1% of scores")
for ax in axes:
    ax.set_xticks(x + w * (len(SHOWN) - 1) / 2, ARMS)
    ax.legend(loc="upper left", bbox_to_anchor=(0, -0.08), ncol=3, frameon=False)
plt.show()

In [ ]:
# Frauds caught (y) when the X% highest-scored test transfers are flagged (x), up to 5%.
# Colour = arm; solid = RUN, dashed = REFERENCE.
def caught(run, arm, upto=0.05):
    pred = pd.read_parquet(RUNS / run / "outputs" / "predictions" / "test.parquet",
                           columns=["label", f"score_{arm}"])
    y = pred.sort_values(f"score_{arm}", ascending=False)["label"].to_numpy()
    n = int(len(y) * upto)
    return np.arange(1, n + 1) / len(y), np.cumsum(y)[:n] / y.sum()

fig, ax = plt.subplots(figsize=(9, 4))
for run in SHOWN:
    if not (RUNS / run / "outputs" / "predictions" / "test.parquet").exists():
        print(f"{run}: no predictions/test.parquet")
        continue
    for i, arm in enumerate(ARMS):
        ax.plot(*caught(run, arm), c=f"C{i}", ls="-" if run == RUN else "--", label=f"{run}: {arm}")
ax.plot([0, 0.05], [0, 0.05], c="gray", ls=":", label="random")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("transfers flagged (highest scores first)")
ax.set_ylabel("frauds caught")
ax.legend()
plt.show()

### Leakage check: does the model recognise customers instead of fraud?

Train and test hold the same customers (the split is by time). If most test frauds are on
customers that already had a fraud in train, the model can score well just by recognising
them. Compare AUC and AP on those customers with the other customers (AP chance = the group's fraud rate):
high on the first group and much lower on the second = the model learned *who*, not *what*.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

train = pd.read_parquet(R / "embeddings" / "train_features.parquet", columns=[CUST, "_target"])
known = set(train.loc[train["_target"] == 1, CUST])     # customers with a fraud in train
pred = pd.read_parquet(R / "outputs" / "predictions" / "test.parquet")
d = test.merge(pred, on="__row_id__")
d["known"] = d[CUST].isin(known)
print(f"test frauds on customers with a fraud in train: {d.loc[d['label'] == 1, 'known'].mean():.1%}\n")

rows = {}
for name, g in (("customers with a fraud in train", d[d["known"]]),
                ("other customers", d[~d["known"]])):
    rows[name] = {"transfers": len(g), "frauds": int(g["label"].sum()), "fraud rate": g["label"].mean()}
    ok = 0 < g["label"].sum() < len(g)                   # AUC and AP need both classes
    for arm in ARMS:
        rows[name][f"{arm} AUC"] = roc_auc_score(g["label"], g[f"score_{arm}"]) if ok else np.nan
        rows[name][f"{arm} AP"] = average_precision_score(g["label"], g[f"score_{arm}"]) if ok else np.nan
fmt(pd.DataFrame(rows).T, {"transfers": "{:,.0f}", "frauds": "{:,.0f}", "fraud rate": "{:.4%}",
                           **{f"{a} {k}": "{:.3f}" for a in ARMS for k in ("AUC", "AP")}})

In [ ]:
# Does the label stick to the customer? Share of transfers flagged on or after each customer's first flag.
# Close to 100% = once flagged, every later transfer is flagged too.
SPLIT_DIR = Experiment.load(R / "experiment.yaml", validate=False).paths()["SPLIT_DIR"]
TIME = SCH["datetime"]
df = pd.concat([pd.read_parquet(SPLIT_DIR / f"{s}.parquet", columns=[CUST, TIME, "_target"])
                for s in ("train", "val", "test")])
first = df[df["_target"] == 1].groupby(CUST)[TIME].min()
after = df[df[TIME] >= df[CUST].map(first)]   # transfers on or after each customer's first flag
print(f"flagged after a customer's first flag: {after['_target'].mean():.1%}")

In [ ]:
# Customers seen in the train period vs customers new in test (for first-flag runs).
# Much worse on new customers = the model relies on knowing the customer.
from sklearn.metrics import average_precision_score, roc_auc_score

SPLIT_DIR = Experiment.load(R / "experiment.yaml", validate=False).paths()["SPLIT_DIR"]
seen = set(pd.read_parquet(SPLIT_DIR / "train.parquet", columns=[CUST])[CUST])
d = test.merge(pd.read_parquet(R / "outputs" / "predictions" / "test.parquet"), on="__row_id__")
rows = {}
for name, g in (("seen in train", d[d[CUST].isin(seen)]), ("new in test", d[~d[CUST].isin(seen)])):
    ok = 0 < g["label"].sum() < len(g)                   # AUC and AP need both classes
    rows[name] = {"customers": g[CUST].nunique(), "frauds": int(g["label"].sum()),
                  "fraud rate": g["label"].mean(),
                  **{f"{a} {k}": (f(g["label"], g[f"score_{a}"]) if ok else np.nan)
                     for a in ARMS for k, f in (("AUC", roc_auc_score), ("AP", average_precision_score))}}
fmt(pd.DataFrame(rows).T, {"customers": "{:,.0f}", "frauds": "{:,.0f}", "fraud rate": "{:.4%}",
                           **{f"{a} {k}": "{:.3f}" for a in ARMS for k in ("AUC", "AP")}})

In [ ]:
# Learned pooling only: the attention head also scores on its own.
if "head" in M:
    display(pd.DataFrame(M["head"]).T)
else:
    print("No learned head in this run (pooling is not 'learned').")

## 4. Per month (test period)

A model that is good in one month and at chance in the others is not stable. Months
with few frauds (`n_pos`) are noisy; do not read much into them. Left: AUC; right: AP. Dashed lines: `REFERENCE`.

In [ ]:
bm_path = R / "outputs" / "metrics_by_month.csv"
if bm_path.exists():
    bm = pd.read_csv(bm_path)
    auc = bm.pivot(index="period", columns="model", values="auc")
    ap = bm.pivot(index="period", columns="model", values="ap")
    xgb = [c for c in ("raw", "embeddings", "combined") if c in auc]
    n_pos = bm.groupby("period")["n_pos"].first().rename("frauds")
    display(pd.concat([n_pos, auc[xgb].round(3).add_suffix(" AUC"), ap[xgb].round(3).add_suffix(" AP")], axis=1))

    ref_path = RUNS / REFERENCE / "outputs" / "metrics_by_month.csv"
    ref = pd.read_csv(ref_path) if REFERENCE != RUN and ref_path.exists() else None
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
    for ax, col, name in ((axes[0], "auc", "AUC"), (axes[1], "ap", "AP")):
        cur = bm.pivot(index="period", columns="model", values=col)
        for i, model in enumerate(xgb):
            ax.plot(cur.index, cur[model], "o-", c=f"C{i}", label=model)
        if ref is not None:
            r = ref.pivot(index="period", columns="model", values=col)
            for i, model in enumerate(xgb):
                ax.plot(r.index, r[model], "--", c=f"C{i}", alpha=0.6, label=f"{REFERENCE}: {model}")
        ax.set_title(f"{RUN}: test {name} per month (XGBoost)")
        ax.tick_params(axis="x", rotation=30)
    axes[0].axhline(0.5, ls="--", c="gray", label="chance")
    axes[1].legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()
else:
    print(f"{bm_path} not found.")

## 5. All finished runs, compared to the reference

Read straight from each run's `outputs/metrics.json` (no need to run `--summary` first).
`Δ` columns are the difference to `REFERENCE`; under ~0.02–0.03 it is noise. The seed
repeats (`seed_*`) show how much `base` moves by chance alone: a variant must move more
than that.

**Last plot:** one point per run, its test AUC minus `REFERENCE`'s. Error bar = the noise
of that gap; grey band = how far the seed repeats moved by chance. A run helps only if
its point and error bar sit clearly right of 0 *and* of the grey band.

In [ ]:
rows = []
for d in sorted(p for p in RUNS.iterdir() if (p / "outputs" / "metrics.json").exists()):
    m = json.loads((d / "outputs" / "metrics.json").read_text())
    row = {"run": d.name}
    for arm in ("raw", "embeddings", "combined"):
        a = m["arms"].get(arm, {})
        row[f"{arm} AUC"] = a.get("test_auc")
        row[f"{arm} AP"] = a.get("test_ap")
        row[f"{arm} lift"] = a.get("test_lift")
    row["± se"] = m["arms"].get("combined", {}).get("test_auc_se")
    rows.append(row)

runs = pd.DataFrame(rows).set_index("run")
if REFERENCE in runs.index:
    for arm in ("embeddings", "combined"):
        for k in ("AUC", "AP"):
            runs[f"Δ {arm} {k}"] = runs[f"{arm} {k}"] - runs.loc[REFERENCE, f"{arm} {k}"]
fmt(runs, {c: ("{:+.3f}" if c.startswith("Δ") else "{:.3f}") for c in runs.columns})

In [ ]:
# Run-to-run noise from the seed repeats (needs base + seed_1 + seed_2).
seeds = [r for r in runs.index if r == "base" or r.startswith("seed_")]
if len(seeds) >= 2:
    spread = runs.loc[seeds, ["embeddings AUC", "combined AUC"]]
    print(f"Runs: {seeds}")
    print((spread.max() - spread.min()).rename("max - min (test AUC)").round(3).to_string())
    print("\nA variant beats base only if its Δ is clearly larger than this spread.")
else:
    print("Seed repeats not finished yet: run experiments/seed_1.yaml and seed_2.yaml.")

In [ ]:
# Every run against REFERENCE. Error bar: noise of the gap (2 x se x sqrt 2, se of combined).
if REFERENCE in runs.index:
    y = np.arange(len(runs))
    fig, axes = plt.subplots(1, 2, figsize=(12, 0.35 * len(runs) + 1.5), sharey=True)
    for ax, arm in zip(axes, ("embeddings", "combined")):
        d = runs[f"Δ {arm} AUC"].astype(float)
        if len(seeds) >= 2:
            ax.axvspan(d[seeds].min(), d[seeds].max(), color="lightgray", label="seed spread")
        ax.errorbar(d, y, xerr=2 * np.sqrt(2) * runs["± se"].astype(float), fmt="o", capsize=3)
        ax.axvline(0, c="k", lw=0.8)
        ax.set_title(f"{arm}: test AUC minus {REFERENCE}")
    axes[0].set_yticks(y, runs.index)
    axes[0].invert_yaxis()
    axes[0].legend()
    plt.show()